In [1]:
import dspy
from dspy.evaluate.evaluate import Evaluate
from dspy.teleprompt import BootstrapFewShot
from datasets import load_dataset
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# Schritt 1: Initialisierung der Modelle und Konfiguration
# Annahme: Lokale Server für Embedding- und Sprachmodelle sind aktiv.
embedder = dspy.Embedder(
    "openai/embeddinggemma-300M-Q8_0.gguf", 
    api_base="http://localhost:8081/v1", 
    api_key="no_key_needed"
)

local_llm = dspy.LM(
    "openai/gemma-3-4b-it-Q4_K_M.gguf", 
    api_base="http://localhost:8080/v1", 
    api_key="no_key_needed",
    temperature=0.1,
    cache=False
)

dspy.configure(lm=local_llm, embedder=embedder)

# Definition der RAG-Komponenten
client = QdrantClient(host="localhost", port=6333)
collection_name = "illuin-conteb-geography"

In [13]:
from datasets import load_dataset

# Laden des Datensatzes
dataset = load_dataset("illuin-conteb/geography", 'documents', split="train")

documents = [item['og_chunk'] for item in dataset]

# Check
# print(len(documents))
# print(documents[:4])

In [14]:
# Erstellen der Qdrant Collection (nur falls sie nicht existiert)
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# Qdrant Client erstellen
client = QdrantClient(host="localhost", port=6333)
embedding_dim = 768 

# Prüfen, ob die Collection existiert
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=embedding_dim,
            distance=Distance.COSINE
        )
    )
    print(f"Collection '{collection_name}' created successfully.")
else:
    print(f"Collection '{collection_name}' already exists.")


Collection 'illuin-conteb-geography' already exists.


In [19]:
# Embedden und Indexieren der Dokumente
embeddings = embedder(documents)
len(embeddings)

2291

In [20]:
from qdrant_client.models import PointStruct

points = [
    PointStruct(id=i, vector=vec, payload={"text": chunk})
    for i, (chunk, vec) in enumerate(zip(all_chunks, embeddings))
]

In [21]:
# Punkte in Qdrant hochladen (in Batches für große Daten)
batch_size = 100
for i in range(0, len(points), batch_size):
    batch = points[i:i+batch_size]
    client.upsert(
        collection_name=collection_name,
        points=batch
    )

print(f"{len(points)} Chunks erfolgreich in Qdrant gespeichert.")

2291 Chunks erfolgreich in Qdrant gespeichert.


In [22]:
class QdrantRetriever(dspy.Retrieve):
    def __init__(self, client, collection_name, embedder, k=3):
        self._client = client
        self._collection_name = collection_name
        self._embedder = embedder
        self._k = k
        super().__init__()

    def forward(self, query_or_queries, k=None):
        k = k if k is not None else self._k
        query_embeddings = self._embedder(query_or_queries)
        results = [
            self._client.query_points(
                collection_name=collection_name,
                query=query_embeddings,
                limit=k,
            ) for emb in query_embeddings
        ]

        
        # Korrekte Extraktion für Batches
        passages = [p.payload["text"] for p in results[0].points]
        return passages

class GenerateAnswer(dspy.Signature):
    """Beantworte die Frage basierend auf dem bereitgestellten Kontext."""
    context = dspy.InputField(desc="Relevante Fakten zur Beantwortung der Frage.")
    question = dspy.InputField(desc="Die ursprüngliche Nutzerfrage.")
    answer = dspy.OutputField(desc="Eine prägnante und faktenbasierte Antwort.")

class RAG(dspy.Module):
    def __init__(self):
        super().__init__()
        self.retriever = QdrantRetriever(client, collection_name, embedder, k=3)
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)
        
    def forward(self, question):
        context = self.retriever(question)
        prediction = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context, answer=prediction.answer)

In [26]:
# Erstellen eines kleinen Trainingsdatensatzes
train_data = [
    {
        'question': "What industries have contributed to the growth of Belfast's services sector?", 
        'answer': "The industries that have contributed to the growth of Belfast's services sector are financial technology (fintech), tourism, and film."},
    {
        'question': "What is the population of Changsha, and how does it rank in terms of livability in China?",  
        'answer': "Changsha has a population of 10,513,100 and is considered the most livable city in China."},
    {
        'question': "When was Kobe founded, and how did it get its name?", 
        'answer': "Kobe was founded in 1889, and its name comes from Kanbe, an archaic title for supporters of the city's Ikuta Shrine."}
]
trainset = [dspy.Example(**x).with_inputs('question') for x in train_data]

# Definition der Validierungsmetrik
# Die Metrik prüft, ob die generierte Antwort die Gold-Antwort enthält.
def validate_answer(example, pred, trace=None):
    return example.answer.lower() in pred.answer.lower()

In [27]:
teleprompter = BootstrapFewShot(metric=validate_answer, max_bootstrapped_demos=2, max_labeled_demos=2)
# Kompilierung des RAG-Moduls
optimized_rag = teleprompter.compile(RAG(), trainset=trainset)

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:43<00:00, 14.49s/it]

Bootstrapped 0 full traces after 2 examples for up to 1 rounds, amounting to 3 attempts.


In [34]:
# Testfrage
question = "Which groups have influenced the city's history, and what was its role in the Kingdom of Hungary?"

# 1. Ausführung der unoptimierten Pipeline
unoptimized_rag = RAG()
prediction_unoptimized = unoptimized_rag(question)
print(f"Frage: {question}")
print(f"Antwort (Unoptimiert): {prediction_unoptimized.answer}")
print("\n--- Prompt (Unoptimiert) ---")

Frage: Which groups have influenced the city's history, and what was its role in the Kingdom of Hungary?
Antwort (Unoptimiert): Budapest's history has been influenced by Austrians, Bulgarians, Croats, Czechs, Germans, Hungarians, Jews, and Slovaks. From 1536 to 1783, it served as the coronation site and legislative center and capital of the Kingdom of Hungary, where eleven Hungarian kings and eight queens were crowned, and most Hungarian parliament assemblies were held.

--- Prompt (Unoptimiert) ---


In [35]:
local_llm.inspect_history(n=1)





[2025-11-14T16:15:44.456486]

System message:

Your input fields are:
1. `context` (str): Relevante Fakten zur Beantwortung der Frage.
2. `question` (str): Die ursprüngliche Nutzerfrage.
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str): Eine prägnante und faktenbasierte Antwort.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## context ## ]]
{context}

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Beantworte die Frage basierend auf dem bereitgestellten Kontext.


User message:

[[ ## context ## ]]
[1] «The city's history has been influenced by people of many nations and religions, including Austrians, Bulgarians, Croats, Czechs, Germans, Hungarians, Jews and Slovaks. It was the coronation site and legislative center and capital of the Kingdom of Hungary from 1536 to 1783; eleven Hung

In [36]:
#  Ausführung der optimierten Pipeline
prediction_optimized = optimized_rag(question)
print(f"\nFrage: {question}")
print(f"Antwort (Optimiert): {prediction_optimized.answer}")
print("\n--- Prompt (Optimiert) ---")


Frage: Which groups have influenced the city's history, and what was its role in the Kingdom of Hungary?
Antwort (Optimiert): The city's history has been influenced by Austrians, Bulgarians, Croats, Czechs, Germans, Hungarians, Jews and Slovaks. It served as the coronation site and legislative center and capital of the Kingdom of Hungary from 1536 to 1783, where eleven Hungarian kings and eight queens were crowned in St Martin's Cathedral.

--- Prompt (Optimiert) ---


In [37]:
local_llm.inspect_history(n=1)





[2025-11-14T16:16:29.019661]

System message:

Your input fields are:
1. `context` (str): Relevante Fakten zur Beantwortung der Frage.
2. `question` (str): Die ursprüngliche Nutzerfrage.
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str): Eine prägnante und faktenbasierte Antwort.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## context ## ]]
{context}

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Beantworte die Frage basierend auf dem bereitgestellten Kontext.


User message:

This is an example of the task, though some input or output fields are not supplied.

[[ ## question ## ]]
When was Kobe founded, and how did it get its name?


Assistant message:

[[ ## reasoning ## ]]
Not supplied for this particular example. 

[[ ## answer ## ]]
Kobe was founded in 1889, and its name comes 